# vLLM Serving

**Stage 5: Inference Optimization - Notebook 57**

This notebook explores vLLM, the dominant framework for high-throughput LLM serving. We'll cover:
- vLLM architecture and design philosophy
- PagedAttention implementation details
- Continuous batching
- Installation and setup
- Serving Llama 2 with vLLM
- Throughput benchmarks
- Production best practices
- Monitoring and observability

In [ ]:
# Note: This notebook demonstrates vLLM concepts and usage
# Some cells require vLLM installation: pip install vllm

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Optional
import time
from dataclasses import dataclass
import json

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 1. vLLM Overview and Architecture

### Historical Context

**Paper**: "Efficient Memory Management for Large Language Model Serving with PagedAttention" (Kwon et al., UC Berkeley, June 2023)

**The Problem (Early 2023)**:
- Existing frameworks (HuggingFace, FasterTransformer) had poor throughput
- Memory fragmentation wasted 50-80% of GPU memory
- Static batching couldn't handle variable-length sequences efficiently
- KV cache management was a major bottleneck

**Key Innovations**:
1. **PagedAttention**: Virtual memory for KV cache
2. **Continuous Batching**: Dynamic request scheduling
3. **Efficient Memory Sharing**: Prefix caching
4. **Optimized Kernels**: Custom CUDA kernels

**Impact**:
- 2-4x throughput vs HuggingFace Text Generation Inference
- Near-zero memory waste
- Became dominant serving framework (2023-2024)
- Influenced TensorRT-LLM, TGI, and others

**Architecture**:
```
┌─────────────────────────────────────────────┐
│          vLLM Architecture                  │
├─────────────────────────────────────────────┤
│                                             │
│  ┌──────────────┐      ┌─────────────────┐ │
│  │ API Server   │─────▶│ LLM Engine      │ │
│  │ (FastAPI/    │      │ (Core Logic)    │ │
│  │  gRPC)       │      │                 │ │
│  └──────────────┘      └────────┬────────┘ │
│                                 │          │
│         ┌───────────────────────┼──────┐   │
│         │                       │      │   │
│  ┌──────▼──────┐   ┌───────────▼────┐ │   │
│  │ Scheduler   │   │ Model Executor │ │   │
│  │ (Continuous │   │ (Runs model)   │ │   │
│  │  Batching)  │   │                │ │   │
│  └──────┬──────┘   └───────┬────────┘ │   │
│         │                  │          │   │
│  ┌──────▼──────────────────▼────────┐ │   │
│  │   Block Manager                  │ │   │
│  │   (PagedAttention)               │ │   │
│  └──────────────────────────────────┘ │   │
│                                       │   │
│  ┌────────────────────────────────┐  │   │
│  │   GPU Memory (KV Cache Blocks) │  │   │
│  └────────────────────────────────┘  │   │
└─────────────────────────────────────┘   │
```

In [ ]:
# Visualize vLLM vs traditional serving
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Traditional approach
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)

# Show memory allocation
positions = [
    (1, 8, 2, 1.5, 'Request 1\n(max_len=2048)\nUsed: 500', '#e74c3c'),
    (4, 8, 2, 1.5, 'Request 2\n(max_len=2048)\nUsed: 300', '#e67e22'),
    (7, 8, 2, 1.5, 'Request 3\n(max_len=2048)\nUsed: 1500', '#f39c12'),
    (1, 2, 2, 4, 'Wasted\n(60%)', '#95a5a6'),
    (4, 2, 2, 4, 'Wasted\n(75%)', '#95a5a6'),
    (7, 2, 2, 4, 'Wasted\n(27%)', '#95a5a6'),
]

for x, y, w, h, label, color in positions:
    rect = plt.Rectangle((x, y), w, h, color=color, alpha=0.6, edgecolor='black')
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, label, ha='center', va='center', fontsize=8, fontweight='bold')

ax.set_title('Traditional: Contiguous Allocation\n60% Memory Wasted!', 
             fontsize=13, fontweight='bold', color='red')
ax.axis('off')

# vLLM with PagedAttention
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)

# Show block-based allocation
block_size = 0.8
colors_blocks = ['#e74c3c', '#e67e22', '#f39c12']
labels_req = ['R1', 'R2', 'R3']

# Request 1: 4 blocks
for i in range(4):
    x = 0.5 + (i % 5) * 1.0
    y = 8.5 - (i // 5) * 1.0
    rect = plt.Rectangle((x, y), block_size, block_size, 
                         color=colors_blocks[0], alpha=0.7, edgecolor='black')
    ax.add_patch(rect)
    ax.text(x + block_size/2, y + block_size/2, 'R1', 
           ha='center', va='center', fontsize=8, fontweight='bold')

# Request 2: 2 blocks
for i in range(2):
    x = 0.5 + (4 + i) * 1.0
    y = 8.5
    rect = plt.Rectangle((x, y), block_size, block_size,
                         color=colors_blocks[1], alpha=0.7, edgecolor='black')
    ax.add_patch(rect)
    ax.text(x + block_size/2, y + block_size/2, 'R2',
           ha='center', va='center', fontsize=8, fontweight='bold')

# Request 3: 12 blocks
for i in range(12):
    x = 0.5 + (i % 5) * 1.0
    y = 7.5 - (i // 5) * 1.0
    if i < 1:  # Skip occupied
        continue
    rect = plt.Rectangle((x, y), block_size, block_size,
                         color=colors_blocks[2], alpha=0.7, edgecolor='black')
    ax.add_patch(rect)
    ax.text(x + block_size/2, y + block_size/2, 'R3',
           ha='center', va='center', fontsize=8, fontweight='bold')

# Free blocks
free_positions = [(0.5 + i*1.0, 4.5) for i in range(5)]
for x, y in free_positions:
    rect = plt.Rectangle((x, y), block_size, block_size,
                         color='white', alpha=0.8, edgecolor='black', linestyle='--')
    ax.add_patch(rect)
    ax.text(x + block_size/2, y + block_size/2, 'Free',
           ha='center', va='center', fontsize=7, style='italic')

ax.set_title('vLLM PagedAttention: Block Allocation\n<5% Memory Wasted!',
             fontsize=13, fontweight='bold', color='green')
ax.axis('off')

plt.tight_layout()
plt.show()

print("Key Improvements:")
print("1. Non-contiguous allocation: No need to reserve max length")
print("2. Fine-grained blocks: Minimal internal fragmentation")
print("3. Dynamic allocation: Grow as sequence grows")
print("4. Memory sharing: Common prefixes share blocks")

## 2. PagedAttention Implementation Details

Let's understand how PagedAttention works under the hood.

In [ ]:
@dataclass
class BlockTable:
    """
    Block table for a sequence.
    Maps logical blocks to physical blocks.
    """
    block_ids: List[int]
    
    def get_physical_block(self, logical_block: int) -> int:
        """Map logical block ID to physical block ID."""
        return self.block_ids[logical_block]
    
    def append_block(self, physical_block: int):
        """Append a new block to the sequence."""
        self.block_ids.append(physical_block)


class BlockAllocator:
    """
    Manages allocation and deallocation of physical blocks.
    Similar to OS page allocator.
    """
    def __init__(self, num_blocks: int):
        self.num_blocks = num_blocks
        self.free_blocks = list(range(num_blocks))
        self.allocated_blocks = {}
    
    def allocate(self, num_blocks: int) -> List[int]:
        """Allocate a list of physical blocks."""
        if len(self.free_blocks) < num_blocks:
            raise RuntimeError(f"Out of memory: need {num_blocks}, have {len(self.free_blocks)}")
        
        allocated = []
        for _ in range(num_blocks):
            block_id = self.free_blocks.pop(0)
            allocated.append(block_id)
        
        return allocated
    
    def free(self, block_ids: List[int]):
        """Free a list of physical blocks."""
        self.free_blocks.extend(block_ids)
    
    def get_num_free_blocks(self) -> int:
        return len(self.free_blocks)


class PagedKVCache:
    """
    PagedAttention KV cache implementation.
    
    Key insight: Store KV cache in fixed-size blocks that can be
    allocated non-contiguously.
    """
    def __init__(
        self,
        num_blocks: int,
        block_size: int,
        num_layers: int,
        num_heads: int,
        head_dim: int,
        dtype: torch.dtype = torch.float16,
    ):
        self.num_blocks = num_blocks
        self.block_size = block_size
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.head_dim = head_dim
        self.dtype = dtype
        
        # Physical KV cache: [num_blocks, num_layers, 2, num_heads, block_size, head_dim]
        # 2 for K and V
        self.cache = torch.zeros(
            num_blocks, num_layers, 2, num_heads, block_size, head_dim,
            dtype=dtype, device=device
        )
        
        # Block allocator
        self.allocator = BlockAllocator(num_blocks)
        
        # Sequence block tables
        self.block_tables: Dict[int, BlockTable] = {}
    
    def allocate_sequence(self, seq_id: int, num_blocks: int) -> BlockTable:
        """Allocate blocks for a new sequence."""
        block_ids = self.allocator.allocate(num_blocks)
        block_table = BlockTable(block_ids)
        self.block_tables[seq_id] = block_table
        return block_table
    
    def free_sequence(self, seq_id: int):
        """Free all blocks for a sequence."""
        if seq_id in self.block_tables:
            block_table = self.block_tables.pop(seq_id)
            self.allocator.free(block_table.block_ids)
    
    def append_slots(self, seq_id: int, num_tokens: int):
        """Append slots for new tokens (allocate new blocks if needed)."""
        block_table = self.block_tables[seq_id]
        current_capacity = len(block_table.block_ids) * self.block_size
        
        # Calculate how many new blocks needed
        blocks_needed = (num_tokens - 1) // self.block_size + 1
        blocks_needed -= len(block_table.block_ids)
        
        if blocks_needed > 0:
            new_blocks = self.allocator.allocate(blocks_needed)
            for block_id in new_blocks:
                block_table.append_block(block_id)
    
    def get_memory_stats(self) -> Dict:
        """Get memory statistics."""
        allocated = self.num_blocks - self.allocator.get_num_free_blocks()
        utilization = allocated / self.num_blocks
        
        cache_size_gb = (
            self.cache.numel() * self.cache.element_size() / 1024**3
        )
        
        return {
            'total_blocks': self.num_blocks,
            'allocated_blocks': allocated,
            'free_blocks': self.allocator.get_num_free_blocks(),
            'utilization': utilization,
            'cache_size_gb': cache_size_gb,
            'active_sequences': len(self.block_tables),
        }

# Example usage
print("PagedAttention KV Cache Example")
print("="*70)

cache = PagedKVCache(
    num_blocks=100,
    block_size=16,
    num_layers=32,
    num_heads=32,
    head_dim=128,
)

print(f"\nInitial state:")
stats = cache.get_memory_stats()
for key, value in stats.items():
    print(f"  {key}: {value}")

# Simulate multiple sequences
sequences = [
    (0, 100),  # seq_id, length
    (1, 200),
    (2, 50),
    (3, 150),
]

print("\nAllocating sequences:")
for seq_id, length in sequences:
    num_blocks_needed = (length + cache.block_size - 1) // cache.block_size
    try:
        block_table = cache.allocate_sequence(seq_id, num_blocks_needed)
        print(f"  Seq {seq_id} (len={length}): allocated {num_blocks_needed} blocks")
        print(f"    Block IDs: {block_table.block_ids}")
    except RuntimeError as e:
        print(f"  Seq {seq_id}: FAILED - {e}")

print("\nAfter allocation:")
stats = cache.get_memory_stats()
for key, value in stats.items():
    print(f"  {key}: {value}")

# Free a sequence
print("\nFreeing sequence 1...")
cache.free_sequence(1)

stats = cache.get_memory_stats()
print(f"  Utilization after free: {stats['utilization']:.1%}")
print(f"  Free blocks: {stats['free_blocks']}")

## 3. Continuous Batching

Traditional batching processes all requests together. Continuous batching adds/removes requests dynamically.

In [ ]:
# Visualize continuous batching
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Static batching
ax = axes[0]
ax.set_xlim(0, 12)
ax.set_ylim(0, 4)

# Show batches
batches = [
    # (start_time, duration, batch_id, requests)
    (0, 3, 'Batch 1', ['R1 (100)', 'R2 (80)', 'R3 (50)']),
    (3, 2, 'Batch 2', ['R4 (60)', 'R5 (40)']),
    (5, 4, 'Batch 3', ['R6 (120)', 'R7 (110)', 'R8 (100)']),
]

colors_batches = ['#e74c3c', '#3498db', '#2ecc71']
for i, (start, duration, batch_id, requests) in enumerate(batches):
    rect = plt.Rectangle((start, 0.5), duration, 3, 
                         color=colors_batches[i], alpha=0.3, edgecolor='black')
    ax.add_patch(rect)
    
    # Show requests
    for j, req in enumerate(requests):
        y = 3.2 - j * 0.8
        ax.text(start + duration/2, y, req, ha='center', va='center', fontsize=9)
    
    ax.text(start + duration/2, 0.2, batch_id, ha='center', va='center', 
           fontsize=10, fontweight='bold')

# Show waiting times
ax.arrow(3.5, 3.5, 1.3, 0, head_width=0.15, head_length=0.1, fc='red', ec='red')
ax.text(4.2, 3.7, 'R4, R5 wait!', ha='center', fontsize=9, color='red', fontweight='bold')

ax.set_xlabel('Time', fontsize=12)
ax.set_title('Static Batching: Must wait for slowest request',
             fontsize=13, fontweight='bold', color='red')
ax.set_yticks([])
ax.grid(True, alpha=0.3, axis='x')

# Continuous batching
ax = axes[1]
ax.set_xlim(0, 12)
ax.set_ylim(0, 4)

# Show dynamic scheduling
requests_continuous = [
    # (start, end, req_name, y_pos)
    (0, 3, 'R1 (100)', 3.2),
    (0, 2.6, 'R2 (80)', 2.4),
    (0, 1.6, 'R3 (50)', 1.6),
    (1.7, 3.7, 'R4 (60)', 1.6),  # Replaces R3
    (2.7, 4.0, 'R5 (40)', 2.4),  # Replaces R2
    (3.1, 6.5, 'R6 (120)', 3.2),  # Replaces R1
    (4.1, 7.6, 'R7 (110)', 2.4),  # Replaces R5
    (3.8, 6.8, 'R8 (100)', 1.6),  # Replaces R4
]

color_req = '#2ecc71'
for start, end, req_name, y in requests_continuous:
    width = end - start
    rect = plt.Rectangle((start, y - 0.3), width, 0.6,
                         color=color_req, alpha=0.6, edgecolor='black')
    ax.add_patch(rect)
    ax.text(start + width/2, y, req_name, ha='center', va='center', fontsize=8)

ax.set_xlabel('Time', fontsize=12)
ax.set_title('Continuous Batching: Add/remove requests dynamically',
             fontsize=13, fontweight='bold', color='green')
ax.set_yticks([])
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\nContinuous Batching Benefits:")
print("1. No waiting for slowest request in batch")
print("2. GPU constantly utilized (no idle time)")
print("3. Lower latency for short requests")
print("4. Higher throughput overall")
print("\nTypical improvement: 2-3x throughput vs static batching")

## 4. Installation and Setup

Let's set up vLLM and configure it for optimal performance.

In [ ]:
print("vLLM Installation Guide")
print("="*70)

installation_guide = """
1. BASIC INSTALLATION:

   # Install from PyPI (simplest)
   pip install vllm
   
   # Or with specific CUDA version
   pip install vllm==0.2.7  # Check latest version
   
   # From source (for latest features)
   git clone https://github.com/vllm-project/vllm.git
   cd vllm
   pip install -e .


2. REQUIREMENTS:

   Hardware:
   - NVIDIA GPU with Compute Capability >= 7.0 (V100+)
   - Recommended: A100, H100 for best performance
   - At least 16GB GPU memory (depends on model)
   
   Software:
   - Python 3.8+
   - CUDA 11.8 or 12.1
   - PyTorch 2.0+


3. QUICK START (Python API):

   from vllm import LLM, SamplingParams
   
   # Initialize model
   llm = LLM(
       model="meta-llama/Llama-2-7b-hf",
       tensor_parallel_size=1,  # Number of GPUs
       dtype="float16",
   )
   
   # Generate
   prompts = ["Hello, how are you?", "What is AI?"]
   sampling_params = SamplingParams(temperature=0.8, top_p=0.95, max_tokens=100)
   outputs = llm.generate(prompts, sampling_params)
   
   for output in outputs:
       print(f"Prompt: {output.prompt}")
       print(f"Output: {output.outputs[0].text}")


4. OPENAI-COMPATIBLE SERVER:

   # Start server
   python -m vllm.entrypoints.openai.api_server \
       --model meta-llama/Llama-2-7b-hf \
       --port 8000 \
       --tensor-parallel-size 1
   
   # Use with OpenAI client
   from openai import OpenAI
   
   client = OpenAI(
       api_key="EMPTY",
       base_url="http://localhost:8000/v1",
   )
   
   completion = client.chat.completions.create(
       model="meta-llama/Llama-2-7b-hf",
       messages=[{"role": "user", "content": "Hello!"}]
   )


5. DOCKER DEPLOYMENT:

   # Pull official image
   docker pull vllm/vllm-openai:latest
   
   # Run server
   docker run --runtime nvidia --gpus all \
       -v ~/.cache/huggingface:/root/.cache/huggingface \
       -p 8000:8000 \
       --ipc=host \
       vllm/vllm-openai:latest \
       --model meta-llama/Llama-2-7b-hf


6. KUBERNETES DEPLOYMENT:

   # See full example in production section below
   # Use Helm chart or custom deployment
   # Configure autoscaling based on queue length
"""

print(installation_guide)

## 5. Serving Llama 2 with vLLM

Complete example of serving Llama 2.

In [ ]:
print("vLLM Configuration for Llama 2")
print("="*70)

# Configuration examples for different scenarios
configs = {
    'development': """
# Development/Testing Configuration
# Single GPU, small model

from vllm import LLM, SamplingParams

llm = LLM(
    model="meta-llama/Llama-2-7b-chat-hf",
    tensor_parallel_size=1,
    dtype="float16",
    gpu_memory_utilization=0.90,  # Use 90% of GPU memory
    max_model_len=2048,           # Context length
    block_size=16,                # PagedAttention block size
)

sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=512,
    stop=["</s>"],
)
""",
    'production_single': """
# Production Configuration (Single GPU)
# Optimized for throughput

from vllm import LLM, SamplingParams

llm = LLM(
    model="meta-llama/Llama-2-13b-chat-hf",
    tensor_parallel_size=1,
    dtype="bfloat16",              # Better than fp16 for training stability
    gpu_memory_utilization=0.95,   # Aggressive memory usage
    max_model_len=4096,
    block_size=16,
    max_num_batched_tokens=8192,   # Continuous batching limit
    max_num_seqs=256,              # Max concurrent sequences
    enable_prefix_caching=True,    # Cache common system prompts
    disable_log_stats=False,       # Keep logging for monitoring
)
""",
    'production_multi': """
# Production Configuration (Multi-GPU)
# For large models (70B+)

from vllm import LLM, SamplingParams

llm = LLM(
    model="meta-llama/Llama-2-70b-chat-hf",
    tensor_parallel_size=4,         # 4x A100 80GB
    pipeline_parallel_size=1,       # Pipeline parallelism (if needed)
    dtype="bfloat16",
    gpu_memory_utilization=0.95,
    max_model_len=4096,
    block_size=16,
    max_num_batched_tokens=16384,   # Larger for multi-GPU
    max_num_seqs=512,               # More concurrent requests
    enable_prefix_caching=True,
    distributed_executor_backend="ray",  # Ray for multi-node
)
""",
    'speculative': """
# Speculative Decoding Configuration
# 2-3x speedup

from vllm import LLM, SamplingParams

llm = LLM(
    model="meta-llama/Llama-2-70b-chat-hf",
    speculative_model="meta-llama/Llama-2-7b-chat-hf",  # Draft model
    num_speculative_tokens=5,       # Draft length
    use_v2_block_manager=True,      # Required for speculation
    tensor_parallel_size=4,
    dtype="bfloat16",
    gpu_memory_utilization=0.90,    # Leave room for draft model
    max_model_len=4096,
)
""",
}

for config_name, config_code in configs.items():
    print(f"\n{'='*70}")
    print(f"{config_name.upper().replace('_', ' ')}")
    print(f"{'='*70}")
    print(config_code)

In [ ]:
# Mock example showing usage patterns
print("\nvLLM Usage Patterns")
print("="*70)

usage_examples = """
1. BASIC GENERATION:

prompts = [
    "Explain quantum computing in simple terms.",
    "Write a Python function to sort a list.",
]

outputs = llm.generate(prompts, sampling_params)

for output in outputs:
    print(f"Prompt: {output.prompt}")
    print(f"Generated: {output.outputs[0].text}")
    print(f"Tokens: {len(output.outputs[0].token_ids)}")
    print()


2. CHAT COMPLETION:

from vllm import LLM, SamplingParams

# Format conversation
conversation = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is machine learning?"},
]

# Apply chat template (model-specific)
prompt = tokenizer.apply_chat_template(
    conversation,
    tokenize=False,
    add_generation_prompt=True
)

output = llm.generate(prompt, sampling_params)


3. STREAMING GENERATION:

# For OpenAI-compatible server
completion = client.chat.completions.create(
    model="meta-llama/Llama-2-7b-chat-hf",
    messages=[{"role": "user", "content": "Tell me a story."}],
    stream=True,
)

for chunk in completion:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)


4. BATCH PROCESSING:

# Process large dataset efficiently
import pandas as pd

df = pd.read_csv("prompts.csv")
prompts = df["prompt"].tolist()

# vLLM automatically batches
outputs = llm.generate(prompts, sampling_params)

df["output"] = [o.outputs[0].text for o in outputs]
df.to_csv("results.csv")


5. WITH PREFIX CACHING:

# Efficient for repeated system prompts
system_prompt = "You are an expert programmer. "

prompts = [
    system_prompt + "Write a sorting algorithm.",
    system_prompt + "Explain recursion.",
    system_prompt + "What is Big O notation?",
]

# System prompt is cached automatically!
outputs = llm.generate(prompts, sampling_params)
"""

print(usage_examples)

## 6. Performance Benchmarks

Let's compare vLLM performance against other frameworks.

In [ ]:
# Simulated benchmark data (based on real vLLM paper results)
benchmark_data = {
    'frameworks': ['HF Transformers', 'TGI', 'FasterTransformer', 'vLLM'],
    'throughput_tokens_per_sec': [150, 550, 680, 1200],
    'latency_p50_ms': [450, 180, 150, 120],
    'latency_p99_ms': [2100, 850, 720, 450],
    'memory_usage_gb': [42, 38, 35, 28],
    'max_batch_size': [8, 32, 48, 128],
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

frameworks = benchmark_data['frameworks']
colors_fw = ['#e74c3c', '#e67e22', '#f39c12', '#2ecc71']

# Throughput
ax = axes[0, 0]
bars = ax.bar(frameworks, benchmark_data['throughput_tokens_per_sec'], 
              color=colors_fw, alpha=0.7)
ax.set_ylabel('Tokens/Second', fontsize=12)
ax.set_title('Throughput Comparison (Llama 2 13B, A100)', 
             fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

# Latency
ax = axes[0, 1]
x = np.arange(len(frameworks))
width = 0.35
ax.bar(x - width/2, benchmark_data['latency_p50_ms'], width, 
       label='P50', color='#3498db', alpha=0.7)
ax.bar(x + width/2, benchmark_data['latency_p99_ms'], width,
       label='P99', color='#e74c3c', alpha=0.7)
ax.set_ylabel('Latency (ms)', fontsize=12)
ax.set_title('Latency Comparison', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(frameworks, rotation=45)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Memory usage
ax = axes[1, 0]
bars = ax.bar(frameworks, benchmark_data['memory_usage_gb'],
              color=colors_fw, alpha=0.7)
ax.set_ylabel('Memory Usage (GB)', fontsize=12)
ax.set_title('Memory Usage Comparison', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)} GB',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

# Max batch size
ax = axes[1, 1]
bars = ax.bar(frameworks, benchmark_data['max_batch_size'],
              color=colors_fw, alpha=0.7)
ax.set_ylabel('Max Batch Size', fontsize=12)
ax.set_title('Maximum Batch Size', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

# Calculate improvements
print("\nvLLM vs Baselines:")
print("="*70)
print(f"\nThroughput improvement:")
for i, fw in enumerate(frameworks[:-1]):
    improvement = benchmark_data['throughput_tokens_per_sec'][-1] / benchmark_data['throughput_tokens_per_sec'][i]
    print(f"  vs {fw}: {improvement:.2f}x")

print(f"\nMemory reduction:")
for i, fw in enumerate(frameworks[:-1]):
    reduction = benchmark_data['memory_usage_gb'][i] / benchmark_data['memory_usage_gb'][-1]
    print(f"  vs {fw}: {reduction:.2f}x less memory")

print(f"\nBatch size increase:")
for i, fw in enumerate(frameworks[:-1]):
    increase = benchmark_data['max_batch_size'][-1] / benchmark_data['max_batch_size'][i]
    print(f"  vs {fw}: {increase:.2f}x larger batches")

## 7. Production Best Practices

In [ ]:
print("vLLM Production Best Practices")
print("="*80)

best_practices = """
1. RESOURCE ALLOCATION:

   GPU Memory:
   ✓ Set gpu_memory_utilization=0.90 (start conservative)
   ✓ Monitor OOM errors, adjust down if needed
   ✓ Reserve 5-10% for CUDA context and overhead
   ✓ Use mixed precision (bfloat16) when possible
   
   CPU Memory:
   ✓ Reserve 2-3x model size for model loading
   ✓ Monitor swap usage (should be zero)
   
   Block Size:
   ✓ Default 16 works well for most cases
   ✓ Smaller (8) for very short sequences
   ✓ Larger (32) for very long sequences


2. CONFIGURATION TUNING:

   Batch Settings:
   max_num_batched_tokens = seq_len * target_batch_size * 2
   max_num_seqs = target_concurrency * 2
   
   Example for 4096 context, 32 batch:
   max_num_batched_tokens = 4096 * 32 * 2 = 262144
   max_num_seqs = 32 * 2 = 64
   
   Context Length:
   ✓ Set max_model_len to actual need (not maximum possible)
   ✓ Shorter contexts = more requests = higher throughput
   
   Parallelism:
   ✓ tensor_parallel_size: Model sharding (use for large models)
   ✓ pipeline_parallel_size: Pipeline parallelism (rarely needed)
   ✓ Prefer tensor parallel over pipeline parallel


3. MONITORING:

   Key Metrics:
   - Throughput (tokens/sec)
   - Latency (P50, P95, P99)
   - GPU utilization (should be >90%)
   - KV cache utilization
   - Request queue length
   - OOM errors
   
   Prometheus Metrics:
   # vLLM exposes metrics on /metrics endpoint
   vllm_num_requests_running
   vllm_num_requests_waiting
   vllm_gpu_cache_usage_perc
   vllm_num_preemptions_total
   vllm_time_to_first_token_seconds
   vllm_time_per_output_token_seconds


4. SCALING:

   Horizontal Scaling:
   ✓ Deploy multiple vLLM instances behind load balancer
   ✓ Use consistent hashing for prefix caching
   ✓ Monitor per-instance metrics
   
   Vertical Scaling:
   ✓ Increase tensor_parallel_size for larger models
   ✓ Use multiple nodes with Ray for very large models
   
   Auto-scaling:
   ✓ Scale based on queue length, not just CPU/GPU
   ✓ Warm-up time can be 30-60s (model loading)
   ✓ Set min replicas to handle baseline traffic


5. RELIABILITY:

   Health Checks:
   # Liveness probe
   GET /health
   
   # Readiness probe
   GET /health (check model loaded)
   
   Error Handling:
   ✓ Timeout requests (set reasonable max_tokens)
   ✓ Retry with exponential backoff
   ✓ Circuit breaker for cascading failures
   ✓ Graceful degradation (fallback to smaller model)
   
   Request Validation:
   ✓ Validate input length (reject oversized)
   ✓ Sanitize inputs
   ✓ Rate limiting per client


6. OPTIMIZATION CHECKLIST:

   ✓ Enable prefix caching for common system prompts
   ✓ Use speculative decoding (2-3x speedup)
   ✓ Tune block_size based on sequence distribution
   ✓ Set appropriate max_model_len (shorter = more throughput)
   ✓ Use bfloat16 when available
   ✓ Enable Flash Attention (automatic in recent versions)
   ✓ Profile with different batch sizes
   ✓ Monitor cache hit rate (prefix caching)
   ✓ Use continuous profiling in production


7. COMMON PITFALLS:

   ✗ Setting max_model_len too large (wastes memory)
   ✗ Not monitoring cache utilization (may need more GPU memory)
   ✗ Using pipeline parallelism when tensor parallel works
   ✗ Ignoring warmup time in autoscaling
   ✗ Not validating input lengths
   ✗ Deploying without load testing
   ✗ Using fp32 (always use fp16/bf16)
"""

print(best_practices)

## 8. Monitoring and Observability

In [ ]:
print("vLLM Monitoring Setup")
print("="*80)

monitoring_guide = """
1. PROMETHEUS + GRAFANA SETUP:

   # docker-compose.yml
   version: '3'
   services:
     vllm:
       image: vllm/vllm-openai:latest
       runtime: nvidia
       environment:
         - VLLM_PORT=8000
       ports:
         - "8000:8000"
       command: >
         --model meta-llama/Llama-2-7b-hf
         --host 0.0.0.0
         --port 8000
     
     prometheus:
       image: prom/prometheus
       ports:
         - "9090:9090"
       volumes:
         - ./prometheus.yml:/etc/prometheus/prometheus.yml
     
     grafana:
       image: grafana/grafana
       ports:
         - "3000:3000"

   # prometheus.yml
   scrape_configs:
     - job_name: 'vllm'
       static_configs:
         - targets: ['vllm:8000']
       metrics_path: '/metrics'
       scrape_interval: 15s


2. KEY DASHBOARDS:

   Throughput Panel:
   rate(vllm_num_requests_finished_total[5m]) * 60
   
   Latency Panel:
   histogram_quantile(0.50, rate(vllm_time_to_first_token_seconds_bucket[5m]))
   histogram_quantile(0.95, rate(vllm_time_to_first_token_seconds_bucket[5m]))
   histogram_quantile(0.99, rate(vllm_time_to_first_token_seconds_bucket[5m]))
   
   Queue Length Panel:
   vllm_num_requests_waiting
   vllm_num_requests_running
   
   Cache Utilization Panel:
   vllm_gpu_cache_usage_perc
   
   Error Rate Panel:
   rate(vllm_request_failure_total[5m]) * 60


3. ALERTING RULES:

   # High latency alert
   - alert: HighLatency
     expr: |
       histogram_quantile(0.95,
         rate(vllm_time_to_first_token_seconds_bucket[5m])
       ) > 1.0
     for: 5m
     annotations:
       summary: "P95 latency > 1s"
   
   # High queue length alert
   - alert: HighQueueLength
     expr: vllm_num_requests_waiting > 100
     for: 2m
     annotations:
       summary: "Request queue > 100"
   
   # OOM alert
   - alert: CacheExhausted
     expr: vllm_gpu_cache_usage_perc > 98
     for: 1m
     annotations:
       summary: "GPU cache nearly full"
   
   # High error rate
   - alert: HighErrorRate
     expr: |
       rate(vllm_request_failure_total[5m]) /
       rate(vllm_num_requests_finished_total[5m]) > 0.05
     for: 5m
     annotations:
       summary: "Error rate > 5%"


4. CUSTOM LOGGING:

   import logging
   import time
   from vllm import LLM, SamplingParams
   
   # Configure logging
   logging.basicConfig(
       level=logging.INFO,
       format='%(asctime)s - %(levelname)s - %(message)s'
   )
   logger = logging.getLogger(__name__)
   
   # Wrap generation with metrics
   def generate_with_metrics(llm, prompts, sampling_params):
       start_time = time.time()
       
       try:
           outputs = llm.generate(prompts, sampling_params)
           
           # Log metrics
           elapsed = time.time() - start_time
           total_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
           throughput = total_tokens / elapsed
           
           logger.info(f"Generation completed: "
                      f"prompts={len(prompts)}, "
                      f"tokens={total_tokens}, "
                      f"time={elapsed:.2f}s, "
                      f"throughput={throughput:.1f} tokens/s")
           
           return outputs
           
       except Exception as e:
           logger.error(f"Generation failed: {e}")
           raise


5. DISTRIBUTED TRACING:

   # OpenTelemetry integration
   from opentelemetry import trace
   from opentelemetry.instrumentation.fastapi import FastAPIInstrumentor
   
   # Instrument FastAPI app (for OpenAI-compatible server)
   FastAPIInstrumentor.instrument_app(app)
   
   # Custom spans
   tracer = trace.get_tracer(__name__)
   
   with tracer.start_as_current_span("llm_generation"):
       outputs = llm.generate(prompts, sampling_params)
"""

print(monitoring_guide)

## Summary

### Key Takeaways

1. **vLLM Architecture**:
   - PagedAttention: Virtual memory for KV cache
   - Continuous batching: Dynamic request scheduling
   - Efficient memory management: Near-zero waste
   - Custom CUDA kernels: Optimized performance

2. **Performance Gains**:
   - 2-4x throughput vs HuggingFace/TGI
   - 30-40% memory reduction
   - 4-16x larger batch sizes
   - 2-3x lower P99 latency

3. **Key Features**:
   - PagedAttention (memory efficiency)
   - Continuous batching (throughput)
   - Prefix caching (repeated prompts)
   - Speculative decoding (2-3x speedup)
   - Multi-GPU support (tensor parallelism)
   - OpenAI-compatible API

4. **Production Setup**:
   - Start with gpu_memory_utilization=0.90
   - Tune max_num_batched_tokens based on workload
   - Enable prefix caching for common prompts
   - Monitor cache utilization and queue length
   - Use continuous profiling

5. **Best Practices**:
   - Set max_model_len to actual need (not maximum)
   - Use bfloat16 for best quality/performance
   - Enable speculative decoding when possible
   - Monitor Prometheus metrics
   - Load test before production
   - Plan for 30-60s warmup time

6. **When to Use vLLM**:
   - High throughput serving (recommended)
   - Variable-length sequences
   - Memory-constrained scenarios
   - Production deployments
   - Multiple concurrent requests

### Historical Impact

- **June 2023**: vLLM paper released (UC Berkeley)
- **2023**: Rapid adoption, became dominant framework
- **2024**: Standard for production LLM serving
- **Influence**: Inspired TensorRT-LLM, TGI improvements

### Comparison to Alternatives

| Framework | Throughput | Memory | Ease of Use | When to Use |
|-----------|------------|--------|-------------|-------------|
| vLLM | Best | Best | High | Production (recommended) |
| TensorRT-LLM | Excellent | Excellent | Medium | NVIDIA-only, ultimate performance |
| TGI | Good | Good | High | Hugging Face ecosystem |
| HF Transformers | Baseline | Baseline | Highest | Development, research |

### Next Steps

- **Notebook 58**: TensorRT-LLM (NVIDIA optimization)
- **Notebook 59**: Continuous Batching (deep dive)